<a href="https://colab.research.google.com/github/Heejung111/-/blob/main/%EA%B0%84%ED%98%B8%EC%A0%95%EB%B3%B4%ED%95%99%EC%8B%A4%EC%8A%B5_%EA%B9%83%ED%97%88%EB%B8%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **구글코랩이 가지고 있는 데이터프레임들**

In [ ]:
!pip list

Package                               Version
------------------------------------- -------------------
absl-py                               1.4.0
accelerate                            1.14.0
access                                1.1.10.post3
affine                                3.0.0
aiofiles                              25.1.0
aiohappyeyeballs                      2.7.1
aiohttp                               3.14.3
aiosignal                             1.4.0
aiosqlite                             0.22.1
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.12.1
altair                                5.5.0
annotated-doc                         0.0.5
annotated-types                       0.8.0
antlr4-python3-runtime                4.9.3
anyio                                 4.14.2
anywidget                             0.9.21
apsw                                  3.53.4.

# **CPU와 GPU 학습속도 비교**

In [ ]:
# ============================================================
# Keras를 이용한 CPU와 GPU 딥러닝 학습 속도 비교
# ============================================================

# 1. 필요한 라이브러리 불러오기
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import time

In [ ]:
# ------------------------------------------------------------
# 2. 현재 GPU가 사용 가능한지 확인
# ------------------------------------------------------------

# TensorFlow가 인식하고 있는 GPU 목록을 출력한다.
# GPU가 정상적으로 연결되어 있다면 GPU 정보가 출력된다.
print("사용 가능한 GPU:")
print(tf.config.list_physical_devices('GPU'))

사용 가능한 GPU:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ------------------------------------------------------------
# 3. 딥러닝 학습에 사용할 가상의 데이터 생성
# ------------------------------------------------------------

# 입력 데이터 50,000개 생성
# 각 데이터는 500개의 숫자(특성)를 가진다.
# 실제 연구에서는 환자의 특성, 검사값, 설문 문항 등이
# 이러한 입력 변수에 해당할 수 있다.

X = np.random.rand(50000, 500).astype("float32") #(50,000개의 사례(행) × 500개의 변수(열))#

# 정답 데이터 생성
# 0 또는 1 중 하나를 가지는 이진분류 데이터이다.
y = np.random.randint(0, 2, size=(50000, 1)).astype("float32")
#np → NumPy, random → 무작위 값을 만드는 기능, randint → random integer, 즉 무작위 정수, (0, 2) → 0 이상, 2 미만의 정수 생성 → 결과는 0 또는 1, size=(50000, 1) → 50,000행 × 1열로 생성, .astype("float32") → 만들어진 0과 1을 32비트 실수형으로 변환#


In [ ]:
# ------------------------------------------------------------
# 4. 딥러닝 모델을 만드는 함수 정의
# ------------------------------------------------------------

#def 는 define의 약자, 파이썬에서 함수(funtion)를 정의할 때 쓰는 명령어, make_model이라는 이름의 함수를 만들겠다

def make_model():

    # Sequential:
    # 신경망의 층(layer)을 순서대로 쌓는 Keras 모델
    # Keras 의 Sequential 방식으로 신경망 모델을 하나 만들어서, 그 모델을 model이라는 변수에 저장한다는 의미#

    model = keras.Sequential([

        # 입력층
        # 하나의 데이터에 500개의 변수가 들어온다는 의미
        layers.Input(shape=(500,)),

        # 은닉층 1
        # 512개의 뉴런을 사용하고 ReLU 활성화 함수를 적용
        layers.Dense(512, activation="relu"),

        # 은닉층 2
        layers.Dense(256, activation="relu"),

        # 은닉층 3
        layers.Dense(128, activation="relu"),

        # 출력층
        # 결과가 0 또는 1인 이진분류이므로 뉴런 1개 사용
        # sigmoid 함수는 결과를 0~1 사이의 확률로 변환
        layers.Dense(1, activation="sigmoid")
    ])

    # 모델 학습 방법 설정
    model.compile(

        # Adam 옵티마이저를 이용하여 가중치를 업데이트
        optimizer="adam",

        # 이진분류에서 사용하는 손실함수
        loss="binary_crossentropy",

        # 정확도도 함께 계산
        metrics=["accuracy"]
    )

    return model



In [ ]:
# ------------------------------------------------------------
# 5. CPU에서 딥러닝 모델 학습
# ------------------------------------------------------------

# /CPU:0을 지정하면 TensorFlow가 CPU를 사용하여 계산한다.
with tf.device("/CPU:0"):

    # 새로운 모델 생성
    cpu_model = make_model()

    # 학습 시작 시간 기록
    start_time = time.time()

    # 모델 학습
    # epochs=5 : 전체 데이터를 5번 반복하여 학습
    # batch_size=256 : 데이터를 한 번에 256개씩 처리
    cpu_model.fit(
        X,
        y,
        epochs=5,
        batch_size=256,
        verbose=0
    )

    # 학습 종료 시간 - 시작 시간
    cpu_time = time.time() - start_time




In [ ]:
# ------------------------------------------------------------
# 6. GPU에서 딥러닝 모델 학습
# ------------------------------------------------------------

# /GPU:0을 지정하면 첫 번째 GPU를 이용하여 계산한다.
with tf.device("/GPU:0"):

    # CPU 실험과 동일한 구조의 새로운 모델 생성
    gpu_model = make_model()

    # 학습 시작 시간 기록
    start_time = time.time()

    # 동일한 데이터와 조건으로 학습
    gpu_model.fit(
        X,
        y,
        epochs=5,
        batch_size=256,
        verbose=0
    )

    # GPU 학습 시간 계산
    gpu_time = time.time() - start_time




In [ ]:
# ------------------------------------------------------------
# 7. CPU와 GPU의 학습 시간 비교
# ------------------------------------------------------------

print("\n===== 학습 속도 비교 =====")

print(f"CPU 학습 시간 : {cpu_time:.2f}초")
print(f"GPU 학습 시간 : {gpu_time:.2f}초")


# CPU 시간이 GPU 시간보다 몇 배 오래 걸렸는지 계산
speedup = cpu_time / gpu_time

print(f"GPU 속도 향상 : 약 {speedup:.2f}배")


===== 학습 속도 비교 =====
CPU 학습 시간 : 13.08초
GPU 학습 시간 : 6.53초
GPU 속도 향상 : 약 2.00배
